# EDM3: Notebook B - API Worker (SLOW / CPU ONLY)

**🚨 IMPORTANT INSTRUCTIONS 🚨**
- Set **Accelerator: None** (We are protecting GPU quota!).
- Ensure `ALPHAGENOME_API_KEY` is added to Kaggle Secrets.
- Go to "Add Data" > "Your Datasets" and mount the `unscored-trajectories` dataset exported from Notebook A to `../input/unscored-trajectories/`.
- Click **Save Version -> Save & Run All (Commit)**. Wait up to 12 hours headlessly.
- If it stops early, just click Save & Run All again. The database is crash-safe and resumes where it left off.
- Do not run interactively. The 12-hour limit requires background execution.

In [ ]:
import os, sys
import subprocess

print("Installing AlphaGenome API client...")
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "alphagenome", "aiohttp"], check=True)
print("Dependencies installed.")

In [ ]:
import numpy as np
import sqlite3
import time
import asyncio
import logging
from tqdm.notebook import tqdm

# ── Global Config ──────────────────────────────────────────────
API_SEQ_LEN     = 131_072       # AlphaGenome required length (N-padded)
API_CONCURRENCY = 2             # Conservative for rate limits (User highly requested respect for rate limit)
API_MAX_RETRIES = 12            # Generous retries for rate limit 429s
API_BASE_BACKOFF = 2.0          # Exponential backoff base

# Paths
INPUT_PATH = "../input/unscored-trajectories/unscored_trajectories.npz"   # Kaggle mounted dataset
DB_PATH    = "/kaggle/working/experience_replay.db" # Incrementally saved WAL DB

from kaggle_secrets import UserSecretsClient
try:
    ALPHAGENOME_API_KEY = UserSecretsClient().get_secret("ALPHAGENOME_API_KEY")
except:
    print("WARNING: Could not load ALPHAGENOME_API_KEY from Kaggle Secrets!")
    ALPHAGENOME_API_KEY = ""

In [ ]:
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s", datefmt="%H:%M:%S")
log = logging.getLogger("api_worker")

def init_database(db_path):
    conn = sqlite3.connect(db_path)
    # Extreme resilience: WAL + synchronous=NORMAL ensures crash safety on headless VM preemption
    conn.execute("PRAGMA journal_mode=WAL")
    conn.execute("PRAGMA synchronous=NORMAL")
    conn.execute("""
        CREATE TABLE IF NOT EXISTS experiences (
            trajectory_id INTEGER PRIMARY KEY,
            actions BLOB NOT NULL,
            forward_log_probs BLOB NOT NULL,
            terminal_onehot BLOB NOT NULL,
            reward REAL NOT NULL,
            api_latency_ms REAL,
            scored_at TEXT DEFAULT CURRENT_TIMESTAMP
        )
    """)
    conn.commit()
    return conn

def get_scored_ids(conn):
    return {r[0] for r in conn.execute("SELECT trajectory_id FROM experiences").fetchall()}

In [ ]:
async def score_sequence(sequence, api_key, semaphore, traj_id):
    """Query AlphaGenome API for DNASE predictions with strict rate limit handling."""
    from alphagenome.models import dna_client

    async with semaphore:
        for attempt in range(API_MAX_RETRIES):
            try:
                loop = asyncio.get_event_loop()
                def _predict():
                    model = dna_client.create(api_key)
                    padded = sequence + 'N' * (API_SEQ_LEN - len(sequence))
                    outputs = model.predict_sequence(
                        sequence=padded,
                        requested_outputs=[dna_client.OutputType.DNASE],
                        ontology_terms=None,
                    )
                    dnase = outputs.dnase
                    if dnase is not None:
                        for attr in ['values', 'data', 'X']:
                            if hasattr(dnase, attr):
                                arr = np.array(getattr(dnase, attr), dtype=np.float32)
                                return arr if arr.ndim >= 2 else arr[:, None]
                    return None

                result = await loop.run_in_executor(None, _predict)
                if result is not None:
                    return result
            except Exception as e:
                msg = str(e)
                if "RESOURCE_EXHAUSTED" in msg or "429" in msg or "Too Many Requests" in msg:
                    # Strict exponential backoff for rate limiting
                    backoff = API_BASE_BACKOFF * (2 ** attempt)
                    log.warning(f"  Traj {traj_id}: Rate limited! 429 received. Backing off {backoff:.0f}s")
                    await asyncio.sleep(backoff)
                else:
                    log.error(f"  Traj {traj_id}: {msg[:120]}")
                    # For other errors, smaller backoff
                    await asyncio.sleep(2.0)
        return None


async def run_scoring(sequences, actions_list, logprobs_list, onehot_list):
    """Score all trajectories asynchronously, respecting constraints."""
    conn = init_database(DB_PATH)
    scored = get_scored_ids(conn)
    pending = [(i, s) for i, s in enumerate(sequences) if i not in scored]
    log.info(f"Scoring: {len(pending)} pending, {len(scored)} already done")

    if not pending:
        log.info("All trajectories already scored! Terminating notebook early.")
        conn.close()
        return

    semaphore = asyncio.Semaphore(API_CONCURRENCY)
    scored_count, failed_count = len(scored), 0
    t0 = time.time()

    pbar = tqdm(total=len(sequences), desc="Scoring Trajectories", initial=len(scored))
    for batch_start in range(0, len(pending), 50):
        batch = pending[batch_start:batch_start + 50]
        tasks = []

        for traj_id, seq in batch:
            tasks.append(score_sequence(seq, ALPHAGENOME_API_KEY, semaphore, traj_id))

        results = await asyncio.gather(*tasks)

        # Commit to disk (WAL) per batch to ensure crash-safety on early termination
        for (traj_id, seq), preds in zip(batch, results):
            if preds is not None:
                mse = float(np.mean(preds ** 2))
                reward = float(np.exp(-mse) + 0.01 * np.mean(np.abs(preds)))
                reward = max(reward, 1e-8)

                conn.execute(
                    "INSERT OR REPLACE INTO experiences "
                    "(trajectory_id, actions, forward_log_probs, terminal_onehot, reward, api_latency_ms) "
                    "VALUES (?, ?, ?, ?, ?, ?)",
                    (traj_id, actions_list[traj_id].tobytes(),
                     logprobs_list[traj_id].tobytes(),
                     onehot_list[traj_id].tobytes(),
                     reward, (time.time() - t0) * 1000 / max(scored_count - len(scored) + 1, 1)),
                )
                scored_count += 1
            else:
                failed_count += 1
        
        # Disk flush
        conn.commit()

        elapsed = time.time() - t0
        pbar.update(len(batch))
        pbar.set_postfix(failed=failed_count)
        # Dual-logging for Kaggle Log Tab visibility
        print(f"> Progress: {scored_count}/{len(sequences)} scored | {failed_count} failed")
        import sys; sys.stdout.flush()
        log.info(f"  [{scored_count}/{len(sequences)}] scored | "
                 f"{failed_count} failed | {elapsed:.0f}s elapsed")

        # Pause slightly between 50-trajectory batches to ease rate limits globally
        await asyncio.sleep(2.0)

    pbar.close()
    log.info(f"Scoring complete: {scored_count} scored, {failed_count} failed")
    conn.close()

In [ ]:
if not ALPHAGENOME_API_KEY:
    raise ValueError("ALPHAGENOME_API_KEY must be set in Kaggle Secrets to run scoring.")
if not os.path.exists(INPUT_PATH):
    raise FileNotFoundError(f"{INPUT_PATH} not found! Did you mount the unscored-trajectories dataset?")

print(f"Loading un-scored trajectories from mounted dataset: {INPUT_PATH}")
data = np.load(INPUT_PATH, allow_pickle=True)
all_seqs = list(data['sequences'])
all_actions = list(data['actions'])
all_logprobs = list(data['forward_log_probs'])
all_onehot = list(data['terminal_onehot'])

print(f"Loaded {len(all_seqs)} trajectories. Starting headless 12-hour asynchronous loop...")

# Run API scoring
await run_scoring(all_seqs, all_actions, all_logprobs, all_onehot)
print("Execution halted (Completion or Timeout).")
